## Load Bronze
Reads the csv file directly from the final_project_raw source schema, writes immutable Parquet snapshots to the food_inspection volume.

In [0]:
catalog_name   = dbutils.widgets.get("catalog_name")
raw_schema  = dbutils.widgets.get("raw_schema")
raw_table  = dbutils.widgets.get("raw_table")
bronze_schema    = dbutils.widgets.get("bronze_schema")
bronze_table = dbutils.widgets.get("bronze_table")
base_path = dbutils.widgets.get("base_path")

print(f"catalog_name   : {catalog_name}")
print(f"raw_schema  : {raw_schema}")
print(f"raw_table  : {raw_table}")
print(f"bronze_schema    : {bronze_schema}")
print(f"bronze_table : {bronze_table}")
print(f"base_path : {base_path}")

catalog_name   : workspace
raw_schema  : final_project_raw
raw_table  : food_inspection
bronze_schema    : final_project_bronze
bronze_table : food_inspection
base_path : /Volumes/workspace/final_project_bronze/food_inspection


In [0]:
from datetime import datetime, timezone
from pyspark.sql import Row

run_ts     = datetime.now(timezone.utc)
run_date   = run_ts.strftime("%Y/%m/%d")
run_ts_str = run_ts.strftime("%Y%m%d_%H%M%S")

### Read Active Tables from Parent Metadata

In [0]:
parent_df = spark.table(f"{catalog_name}.{bronze_schema}.pipeline_metadata_parent")
active_tables = (
    parent_df
    .filter("active_flag = 'Y'")
    .select("table_name", "file_name")
    .collect()
)

print(f"Tables to load into Raw: {[r.table_name for r in active_tables]}")

Tables to load into Raw: ['Food_Inspections_Chicago', 'Food_Inspections_Dallas']


### Extract from Source + Write Parquet Snapshots + Log Metrics

In [0]:
child_rows = []

for row in active_tables:
    table_name = row.table_name
    file_name  = row.file_name
    status     = "FAILED"
    src_count  = 0
    tgt_count  = 0
    file_loc   = ""

    try:
        # load the file
        df = spark \
            .read.format("csv") \
            .option("inferSchema","true") \
            .option("header","true") \
            .option("sep",",") \
            .load(f"/Volumes/{catalog_name}/{raw_schema}/{raw_table}/{table_name}.csv")

        src_count = df.count()

        # Dynamic path — never overwrites previous runs
        file_loc  = f"{base_path}/{table_name.lower()}/{run_date}/{table_name.lower()}_{run_ts_str}.parquet"

        #df.write.mode("overwrite").parquet(file_loc)
        df.write.mode("append").parquet(file_loc)

        tgt_count = spark.read.parquet(file_loc).count()

        if src_count != tgt_count:
            raise ValueError(f"Row count mismatch — source: {src_count}, target: {tgt_count}")

        status = "SUCCESS"
        print(f"  {table_name}: {src_count} rows → {file_loc}")

    except Exception as e:
        status = "FAILED"
        print(f"  {table_name}: FAILED — {e}")

    child_rows.append(Row(
        table_name       = table_name,
        execution_time   = run_ts,
        status           = status,
        source_row_count = src_count,
        target_row_count = tgt_count,
        file_location    = file_loc,
        created_date     = run_ts.date()
    ))

  Food_Inspections_Chicago: 308357 rows → /Volumes/workspace/final_project_bronze/food_inspection/food_inspections_chicago/2026/04/08/food_inspections_chicago_20260408_230522.parquet
  Food_Inspections_Dallas: 78984 rows → /Volumes/workspace/final_project_bronze/food_inspection/food_inspections_dallas/2026/04/08/food_inspections_dallas_20260408_230522.parquet


### Write to Child Execution Metrics Table

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    IntegerType, DateType
)

schema = StructType([
    StructField("table_name",       StringType(),    True),
    StructField("execution_time",   TimestampType(), True),
    StructField("status",           StringType(),    True),
    StructField("source_row_count", IntegerType(),   True),
    StructField("target_row_count", IntegerType(),   True),
    StructField("file_location",    StringType(),    True),
    StructField("created_date",     DateType(),      True),
])

metrics_df = spark.createDataFrame(child_rows, schema)
metrics_df.write.format("delta").mode("append").saveAsTable(
    f"{catalog_name}.{bronze_schema}.pipeline_metadata_child"
)

print("\nChild metrics written.")
display(metrics_df)


Child metrics written.


table_name,execution_time,status,source_row_count,target_row_count,file_location,created_date
Food_Inspections_Chicago,2026-04-08T23:05:22.917Z,SUCCESS,308357,308357,/Volumes/workspace/final_project_bronze/food_inspection/food_inspections_chicago/2026/04/08/food_inspections_chicago_20260408_230522.parquet,2026-04-08
Food_Inspections_Dallas,2026-04-08T23:05:22.917Z,SUCCESS,78984,78984,/Volumes/workspace/final_project_bronze/food_inspection/food_inspections_dallas/2026/04/08/food_inspections_dallas_20260408_230522.parquet,2026-04-08


### Fail the notebook if any table failed

In [0]:
failures = [r for r in child_rows if r.status == "FAILED"]
if failures:
    failed_names = [r.table_name for r in failures]
    raise Exception(f"Raw load failed for tables: {failed_names}")

print("All tables loaded to Raw successfully.")

All tables loaded to Raw successfully.
